In [1]:
import uuid
from importlib.metadata import metadata

import pinecone
from datasets import load_dataset
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer

In [2]:
load_dotenv(find_dotenv(), override=True)

True

In [3]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"), endpoint=os.getenv("PINECONE_ENV"))

In [4]:
fw = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [6]:
pc.create_index(
    name="text",
    dimension=model.get_sentence_embedding_dimension(),
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    )
)

{
    "name": "text",
    "metric": "cosine",
    "host": "text-5blbnge.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [7]:
index = pc.Index("text")

In [8]:
subset_size = 1000

In [9]:
vectors_to_upsert = []
for i, item in enumerate(fw):
    if i >= subset_size:
        break
    text = item["text"]
    unique_id = str(item["id"])
    language = item["language"]
    embeddings = model.encode(text, show_progress_bar=False).tolist()
    metadata = {'language': language}

    vectors_to_upsert.append((unique_id, embeddings, metadata))

In [22]:
batch_size = 100
for i in range(0, len(vectors_to_upsert), batch_size):
    index.upsert(vectors_to_upsert[i:i+batch_size])